# 用一个 MLP 完成 Fashion-MNIST 分类

这个 notebook 的目标不是只跑出一个准确率，而是建立一条完整的深度学习心智模型：**数据如何进入模型、模型如何产生预测、损失如何变成参数更新，以及训练好的参数如何用于新样本**。

学习路线：数据集与张量形状 → `DataLoader` 小批量 → MLP 前向传播 → 交叉熵与反向传播 → 训练/评估模式 → 保存、加载与推理。运行每个代码单元前，先猜一下它的输入、输出和张量形状。

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

## 1. 数据：把图片变成模型能计算的张量

Fashion-MNIST 的每张图片是 28×28 的灰度图，标签是 0–9 的整数类别。`train=True` 的 60,000 张图片只用于学习参数；`train=False` 的测试集在训练期间不参与更新，用来估计泛化能力。

`ToImage()` 把原始样本转换为图像张量，`ToDtype(torch.float32, scale=True)` 将像素从整数转换为 `[0, 1]` 的浮点数。归一化让不同批次的数值尺度稳定，通常能让优化更容易。

In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100%|██████████| 26.4M/26.4M [23:42<00:00, 18.6kB/s]  
100%|██████████| 29.5k/29.5k [00:00<00:00, 100kB/s]
100%|██████████| 4.42M/4.42M [00:05<00:00, 874kB/s] 
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.13MB/s]


In [3]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


### 读懂这个 batch 的形状

输出 `[N, C, H, W] = [64, 1, 28, 28]`：64 是一次取出的样本数，1 是灰度通道，后面是高和宽；标签形状 `[64]` 表示每张图对应一个整数标签。`DataLoader` 把单个样本组成 batch，让模型一次处理一批样本。

**检查点：** 如果把 `batch_size` 改成 32，哪些维度会变化？最后一个 batch 为什么可能不足 32？

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using mps device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## 2. 模型：从二维图片到十个类别分数

`Flatten` 把每张 `1×28×28` 图片展平为 784 个特征。随后数据经过两个隐藏层（`784 → 512 → 512`）和 ReLU，最后得到 10 个 logits。logits 是未归一化的类别分数，并不是概率。

`forward` 描述一次前向传播。`.to(device)` 把参数移动到 CPU 或可用加速器；稍后的输入也必须在同一设备上。

In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

## 3. 学习规则：损失函数和优化器

`CrossEntropyLoss` 比较 logits 与真实标签，输出一个标量损失；SGD 根据损失对参数的梯度，以 `lr=1e-3` 的步长更新参数。交叉熵内部已经包含适合 logits 的归一化计算，因此模型末层不需要手动加 Softmax。

一次更新的顺序是：`loss.backward()` 计算梯度 → `optimizer.step()` 更新参数 → `optimizer.zero_grad()` 清空梯度。PyTorch 默认累加梯度，漏掉清零会把不同 batch 的梯度意外叠加。

In [6]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

## 4. 训练循环：一个 batch 如何改变模型

每轮循环的数据流是：`X → model(X) → pred → loss → gradients → updated parameters`。`model.train()` 打开训练模式；本模型没有 Dropout 或 BatchNorm，所以暂时看不出差别，但养成显式切换模式的习惯很重要。

注意，`loss.item()` 只取出用于显示的 Python 数值，不参与反向传播。打印的 loss 是当前 batch 的损失，不是从 epoch 开始到当前的平均值。

In [7]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

### 评估：测量能力，但不学习

`model.eval()` 切换到评估模式，`torch.no_grad()` 关闭梯度记录，减少内存和计算开销。评估函数不调用优化器，因此测试数据不会改变参数。

`argmax(1)` 在每个样本的 10 个 logits 中取最高分的类别；与标签逐项比较后求和得到正确数量。平均 loss 按 batch 平均，accuracy 则按样本数平均。

In [8]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.306118  [   64/60000]
loss: 2.293521  [ 6464/60000]
loss: 2.268611  [12864/60000]
loss: 2.266574  [19264/60000]
loss: 2.249774  [25664/60000]
loss: 2.225859  [32064/60000]
loss: 2.242353  [38464/60000]
loss: 2.207340  [44864/60000]
loss: 2.216925  [51264/60000]
loss: 2.176348  [57664/60000]
Test Error: 
 Accuracy: 41.6%, Avg loss: 2.170339 

Epoch 2
-------------------------------
loss: 2.184888  [   64/60000]
loss: 2.178230  [ 6464/60000]
loss: 2.114702  [12864/60000]
loss: 2.131805  [19264/60000]
loss: 2.079446  [25664/60000]
loss: 2.026440  [32064/60000]
loss: 2.069047  [38464/60000]
loss: 1.984000  [44864/60000]
loss: 2.010154  [51264/60000]
loss: 1.930698  [57664/60000]
Test Error: 
 Accuracy: 48.0%, Avg loss: 1.924914 

Epoch 3
-------------------------------
loss: 1.960760  [   64/60000]
loss: 1.941372  [ 6464/60000]
loss: 1.811902  [12864/60000]
loss: 1.851887  [19264/60000]
loss: 1.742378  [25664/60000]
loss: 1.686768  [32064/600

### Epoch：反复查看整份训练集

一个 epoch 表示模型遍历一次训练集。每个 epoch 后测试一次，可以观察训练是否真正改善了未参与更新的数据。理想情况下 loss 下降、accuracy 上升；如果训练表现持续提升而测试表现恶化，通常是过拟合信号。

**动手练习：** 先记录 5 个 epoch 的结果，再把学习率改为 `1e-2` 或 `1e-4` 重新初始化模型并训练。比较收敛速度，解释为什么不能只凭某一个 batch 的 loss 下结论。

In [9]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


## 5. 保存与加载：持久化的是学到的参数

`state_dict()` 是参数名到张量的映射。这里只保存权重，不保存 Python 类定义、优化器状态或训练进度。因此加载时必须先创建结构完全一致的 `NeuralNetwork`，再把参数填进去。

若要中断后继续训练，通常还应保存 optimizer state、epoch 和随机状态；这里只有推理需求，所以权重文件已经足够。

In [10]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

## 6. 单样本推理：从分数还原成人类可读类别

数据集中单张图片的形状是 `[1, 28, 28]`，缺少显式 batch 维。不过 `Flatten` 默认保留第 0 维，因此这里仍得到形如 `[1, 10]` 的输出。`pred[0].argmax(0)` 找到最高分的类别索引，再用 `classes` 映射为名称。

**调试提示：** 如果出现设备不一致错误，检查模型和 `x` 是否都调用了 `.to(device)`；如果加载时报键名或尺寸不匹配，检查保存与加载时的模型结构是否相同。

In [11]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


## 总结与进一步练习

现在你应该能沿着完整链路解释：图片被转成浮点张量并组成 batch；MLP 输出 logits；交叉熵衡量预测误差；自动微分产生梯度；SGD 更新参数；测试阶段只测量不更新；`state_dict` 让学到的参数可以复用。

可以继续做三个小实验：

1. 给训练 `DataLoader` 加上 `shuffle=True`，思考每个 epoch 改变样本顺序为什么通常有利。
2. 打印一次前向传播中展平前、展平后和 logits 的 shape，验证 `64×1×28×28 → 64×784 → 64×10`。
3. 将隐藏层宽度从 512 改为 128，比较参数量、训练速度和准确率。成功标准不是一定更准，而是能用“模型容量”解释观察到的变化。